# AI-Based Customer Spending Prediction
**Veda Technology — Day 4 of 4 project**

This notebook demonstrates an end-to-end classification workflow on a synthetic mall-customer dataset: data checks, EDA, feature preparation, two models, evaluation, feature interpretation, and a sample prediction.

> Educational note: the dataset and spending labels are synthetic. Results are not evidence of real-world customer behavior.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

ROOT = Path.cwd()
if not (ROOT / "data" / "mall_customers.csv").exists() and (ROOT.parent / "data" / "mall_customers.csv").exists():
    ROOT = ROOT.parent
df = pd.read_csv(ROOT / "data" / "mall_customers.csv")
df.head()

## 1. Data quality and overview

In [ ]:
print("Shape:", df.shape)
display(df.info())
display(df.describe(include="all").T)
print("Missing values by column:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Target distribution:\n", df["AnnualSpendingCategory"].value_counts())

## 2. Exploratory data analysis — at least five visualizations

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x="AnnualSpendingCategory", order=["Low","Medium","High"])
plt.title("Customer Count by Spending Category"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(data=df, x="Age", hue="AnnualSpendingCategory", multiple="stack", bins=18)
plt.title("Age Distribution by Spending Category"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7,4))
sns.boxplot(data=df, x="AnnualSpendingCategory", y="AnnualIncome_k", order=["Low","Medium","High"])
plt.title("Annual Income by Spending Category"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7,4))
sns.scatterplot(data=df, x="MonthlyVisits", y="AverageBasketValue", hue="AnnualSpendingCategory", alpha=.7)
plt.title("Visits vs Average Basket Value"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8,5))
numeric_df = df.select_dtypes(include="number")
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Numeric Feature Correlation"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(7,4))
sns.boxplot(data=df, x="AnnualSpendingCategory", y="MonthlyVisits", order=["Low","Medium","High"])
plt.title("Monthly Visits by Spending Category"); plt.tight_layout(); plt.show()

## 3. Preprocessing and train/test split

In [ ]:
target = "AnnualSpendingCategory"
X = df.drop(columns=[target, "CustomerID"])
y = df[target]
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 4. Train and compare two classifiers

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42,
                                            class_weight="balanced", min_samples_leaf=2)
}
fitted_models, metrics = {}, {}
for name, estimator in models.items():
    pipe = Pipeline([("preprocess", preprocessor), ("model", estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted_models[name] = pipe
    metrics[name] = accuracy_score(y_test, pred)
    print(f"\n{name} — accuracy: {metrics[name]:.3f}")
    print(classification_report(y_test, pred, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, xticks_rotation=25)
    plt.title(f"{name} Confusion Matrix"); plt.tight_layout(); plt.show()
display(pd.DataFrame({"Model": list(metrics.keys()), "Accuracy": list(metrics.values())}).sort_values("Accuracy", ascending=False))

## 5. Feature importance / interpretation

In [ ]:
rf = fitted_models["Random Forest"]
feature_names = rf.named_steps["preprocess"].get_feature_names_out()
importances = rf.named_steps["model"].feature_importances_
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": importances}).sort_values("Importance", ascending=False)
display(importance_df.head(15))
plt.figure(figsize=(9,6))
sns.barplot(data=importance_df.head(12), x="Importance", y="Feature")
plt.title("Top 12 Random Forest Feature Importances"); plt.tight_layout(); plt.show()

## 6. Sample prediction for a new customer

In [ ]:
new_customer = pd.DataFrame([{
    "Age": 32,
    "Gender": "Female",
    "AnnualIncome_k": 68.0,
    "MonthlyVisits": 7,
    "AverageBasketValue": 2200,
    "OnlinePurchaseShare": 0.45,
    "LoyaltyYears": 3.0
}])
best_model_name = max(metrics, key=metrics.get)
prediction = fitted_models[best_model_name].predict(new_customer)[0]
print("Selected model:", best_model_name)
print("Predicted spending category:", prediction)

## 7. Business insights and limitations
- Visit frequency and average basket value are intuitive purchasing signals to inspect alongside income and loyalty.
- Use the feature-importance chart as a model diagnostic, not as proof of causation.
- Labels were created from a synthetic spending proxy and divided into three quantile-based categories.
- Real deployment requires a representative, consented dataset, leakage checks, fairness evaluation, calibration, and monitoring.
- Avoid using sensitive characteristics for consequential decisions.